# Feature selection pipeline runs (V6)

This notebook drives `run_selection.py` over ablation variants **C0–C5** on `derived_8.0` and `derived_8.2`. Selection uses **train only** (val/test are passed for optional internal metrics but ranking stages fit on train). Artifacts are versioned under `artifacts/<dataset>/<variant>/selected_features.json`.


In [ ]:
from pathlib import Path
import sys
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / "data" / "splits").is_dir() and (p / "Modeling").is_dir():
        PROJECT_ROOT = p
        break
sys.path.insert(0, str(PROJECT_ROOT))

EXP_DIR = PROJECT_ROOT / "notebooks" / "experiment" / "derived_8.2-feature-selection-2.0"
RUNNER = EXP_DIR / "run_selection.py"
assert RUNNER.exists(), RUNNER
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUNNER:", RUNNER)


## Smoke test (fast)

Run the primary **c2_xgb** variant with few bootstraps to validate wiring before the full sweep.


In [ ]:
# Smoke: both datasets, c2 only, n_boot=8
cmd = [
    sys.executable, str(RUNNER),
    "--variants", "c2_xgb",
    "--dataset", "derived_8.0",
    "--dataset", "derived_8.2",
    "--n-boot", "8",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
print(proc.stdout[-4000:] if proc.stdout else "")
if proc.returncode != 0:
    print(proc.stderr[-4000:])
    raise RuntimeError(f"run_selection failed with code {proc.returncode}")
print("Smoke OK")


## Full ablation sweep

Runs all variants C0–C5 on both datasets with the config default of 50 stability bootstraps. This is compute-heavy (especially XGB/RF stability). Re-run only when needed; artifacts are overwritten per variant.


In [ ]:
RUN_FULL = True  # set False to skip after smoke

if RUN_FULL:
    cmd = [sys.executable, str(RUNNER)]  # all variants, both datasets, config n_boot
    print("Running full sweep (this may take a while)...")
    proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
    print(proc.stdout[-6000:] if proc.stdout else "")
    if proc.returncode != 0:
        print(proc.stderr[-6000:])
        raise RuntimeError(f"full sweep failed: {proc.returncode}")
    print("Full sweep complete")
else:
    print("Skipped full sweep (RUN_FULL=False)")


## Artifact summary

Inspect family counts and coverage flags for every produced selection.


In [ ]:
import pandas as pd
from Modeling.Src.soilmoist_fl.Selectors.family_coverage import group_by_coverage_family

rows = []
art_root = EXP_DIR / "artifacts"
for path in sorted(art_root.glob("*/*/selected_features.json")):
    payload = json.loads(path.read_text())
    feats = payload["features"]
    groups = group_by_coverage_family(feats)
    rows.append({
        "dataset": payload["dataset"],
        "variant": payload["variant"],
        "n": payload["n_features"],
        "satellite": len(groups.get("satellite", [])),
        "hydro": len(groups.get("hydro", [])),
        "static": len(groups.get("static", [])),
        "calendar": len(groups.get("calendar", [])),
        "temporal": len(groups.get("temporal", [])),
        "path": str(path.relative_to(PROJECT_ROOT)),
    })

summary = pd.DataFrame(rows)
if len(summary):
    display(summary)
    summary.to_csv(art_root / "artifact_summary.csv", index=False)
else:
    print("No artifacts found yet.")
